# Проверка динамик пары мёбиус-колец (3 ноя 2026)
Каждый раздел: вывод (Markdown) → численная проверка, печатает **OK/FAIL**.
Ядро SageMath 9.6 (код на numpy, работает и в Python 3).
Обозначения: кольцо радиуса $R$ в плоскости $z=sH$, $s=+1$ (N), $s=-1$ (S); базис $(e_r,e_\tau,e_z)$, $e_\tau$ — касательная к линии;
$\varphi=\omega t$; $\chi$ — азимут от особой точки (A, C) или от разреза (B); плоскость поворота $b$: $(u,v)\in\{(r,z),(r,\tau),(z,\tau)\}$.
$$b=s\,b_0(\cos\psi\,u+\sin\psi\,v),\quad \psi_A=\tfrac{\chi}{2}+\tfrac{\varphi}{2},\ \ \psi_B=-\tfrac{\chi}{2}+\tfrac{\varphi}{2},\ \ \psi_C=\tfrac{\chi}{2}+\varphi .$$

In [ ]:
import numpy as np
R, H, B0 = 4.0, 3.0, 1.0
TW = 2*np.pi
m = lambda x: np.mod(x, TW)
def fr(th):
    th = np.asarray(th, float)
    return {'r': np.stack([np.cos(th), np.sin(th), 0*th], -1),
            't': np.stack([-np.sin(th), np.cos(th), 0*th], -1),
            'z': np.stack([0*th, 0*th, 1+0*th], -1)}
PL = {'rz': ('r','z'), 'rt': ('r','t'), 'tz': ('t','z')}
def burgers(kind, s, th, phi, pl, alpha=None):
    # alpha — азимут особой точки при phi=0 (по умолчанию: N в 0, S в π — противофаза)
    if alpha is None: alpha = 0.0 if s > 0 else np.pi
    f = fr(th); u, v = PL[pl]
    if kind == 'A': p = m(th - phi - alpha)/2 + phi/2
    elif kind == 'B': p = -m(th - alpha)/2 + phi/2
    else: p = m(th - phi - alpha)/2 + phi
    return s*B0*(np.cos(p)[..., None]*f[u] + np.sin(p)[..., None]*f[v])
def report(name, ok): print(('OK   ' if ok else 'FAIL ') + name); return ok
TH = np.linspace(0.003, TW - 0.003, 721)

## 1. Период и статистика
Для A и B при $\varphi\to\varphi+2\pi$ получаем $\psi\to\psi+\pi$, т.е. $b\to-b$ (спинорная двузначность, статья: «Период пары и статистика»), возврат за $4\pi$.
Для C: $\psi\to\psi+2\pi$ за один оборот — бозон.

In [ ]:
for k in 'ABC':
    for pl in PL:
        b0 = burgers(k, 1, TH, .3, pl); b2 = burgers(k, 1, TH, .3+TW, pl); b4 = burgers(k, 1, TH, .3+2*TW, pl)
        exp2 = -b0 if k in 'AB' else b0
        report(f'{k} {pl}: φ+2π → {"−b" if k in "AB" else "+b"},  φ+4π → b', np.allclose(b2, exp2) and np.allclose(b4, b0))

## 2. Шов и непрерывная прецессия
A и C: скачок $|\Delta b|=2b_0$ на бегущей особой точке. B: в фиксированной точке $b$ вращается непрерывно с частотой $\omega/2$.

In [ ]:
e = 1e-7
for k in 'AC':
    j = burgers(k, 1, np.array([.3 - e + 0]), .3, 'rz')[0] - burgers(k, 1, np.array([.3 + e]), .3, 'rz')[0]
    report(f'{k}: |Δb| = 2b0 на особой точке', abs(np.linalg.norm(j) - 2*B0) < 1e-5)
ph = np.linspace(0, 2*TW, 801); th0 = 1.0
bb = np.array([burgers('B', 1, np.array([th0]), p, 'rz')[0] for p in ph]); f0 = fr(np.array([th0]))
ang = np.unwrap(np.arctan2(bb @ f0['z'][0], bb @ f0['r'][0]))
report('B: dψ/dφ = 1/2 в фиксированной точке', np.allclose(np.diff(ang), (ph[1]-ph[0])/2, atol=1e-9))

## 3. Суммарный вектор Бюргерса пары (контур вокруг обоих колец)
При сдвиге особой точки S на $\Delta$ относительно N сумма $b_N+b_S$ в одной азимутальной точке есть разность двух единичных векторов с углом между ними $\Delta/2$ или $\Delta/2-\pi$ (в зависимости от дуги), откуда
$$\max_\theta|b_N+b_S| = 2b_0\max\left(|\cos\tfrac{\Delta}{4}|,\ |\sin\tfrac{\Delta}{4}|\right)\ \ (\Delta\neq0),\qquad =0\ \ (\Delta=0).$$
При противофазе $\Delta=\pi$: $|b_N+b_S|=\sqrt2\,b_0$ **во всех точках кольца** — пара не нейтральна.

In [ ]:
phis = np.linspace(0, 2*TW, 25)
def maxsum(k, rings, pl='rz'):
    return max(np.abs(sum(burgers(k, s, TH, p, pl, a) for s, a in rings)).max() for p in phis)
for D in (45, 90, 180, 270):
    d = np.radians(D); num = maxsum('A', [(1, 0.), (-1, d)]); an = 2*B0*max(abs(np.cos(d/4)), abs(np.sin(d/4)))
    report(f'Δ={D}°: max|ΣB| = {num:.3f} = формула {an:.3f}', abs(num - an) < 2e-2)
report('Δ=0°: ΣB = 0', maxsum('A', [(1, 0.), (-1, 0.)]) < 1e-9)
for k in 'ABC':
    for pl in PL:
        report(f'{k} {pl}: противофаза → |ΣB| = √2·b0 всюду', abs(maxsum(k, [(1,0.),(-1,np.pi)], pl) - np.sqrt(2)*B0) < 1e-6)

## 4. Остаток не гасится одним дефектом Вольтерры (deWit 1973 II, ур. 5.5)
Классический дефект добавляет $a+\Omega\times x$ (гармоники по азимуту $n\le1$). Подгонка МНК остатка $S(\theta)$: если остаток после подгонки сравним с $|S|$, компенсация невозможна.

In [ ]:
def volterra_fit(S):
    X = R*fr(TH)['r']; A = np.zeros((3*len(TH), 6))
    for i, x in enumerate(X):
        A[3*i:3*i+3, :3] = np.eye(3)
        A[3*i:3*i+3, 3:] = np.array([[0, x[2], -x[1]], [-x[2], 0, x[0]], [x[1], -x[0], 0]])
    sol = np.linalg.lstsq(A, S.reshape(-1), rcond=None)[0]
    return np.abs((A@sol).reshape(-1, 3) - S).max()
for k in 'AB':
    for pl in PL:
        S = burgers(k, 1, TH, .3, pl) + burgers(k, -1, TH, .3, pl)
        r = volterra_fit(S)
        report(f'{k} {pl}: остаток после a+Ω×x = {r:.2f} b0 (не мал → одной линией не погасить)', r > 0.5*B0)

## 5. Нейтральность и поперечный диполь — теорема
$\sum_k b_k(\theta)=0$ при всех $\theta,\varphi$ ⇔ в каждой азимутальной позиции особых точек $\sum s_k=0$.
**При равных радиусах колец** тогда поперечный дипольный момент особых точек $\sum_k s_k R\,e_r(\varphi+\alpha_k)=0$: остаётся только **осевой** (статический) момент.
⇒ нейтральная конфигурация не имеет вращающегося поперечного диполя (и дипольного излучения, ср. $\tau\sim10^{-18}$ с).
(Исправлено 3 ноя, разд. 9: при разных радиусах это неверно.) Три кольца нейтральными быть не могут ($\sum s_k$ при нечётном числе ≠ 0).

In [ ]:
import itertools
def dip(rings, zs, phi):
    return sum(s*np.array([R*np.cos(phi+a), R*np.sin(phi+a), z]) for (s, a), z in zip(rings, zs))
grid = np.radians(np.arange(0, 360, 45)); found = {3: 0, 4: 0}; transverse_ok = True
for N in (3, 4):
    for signs in itertools.product((1, -1), repeat=N):
        for al in itertools.product(grid, repeat=N):
            pos = {}
            for s, a in zip(signs, al): pos[round(np.degrees(a)) % 360] = pos.get(round(np.degrees(a)) % 360, 0) + s
            if all(v == 0 for v in pos.values()):
                found[N] += 1
                zs = [H*(1 - 0.1*i)*s for i, s in enumerate(signs)]
                transverse_ok &= np.allclose(dip(list(zip(signs, al)), zs, .7)[:2], 0)
report('3 кольца: нейтральных конфигураций нет', found[3] == 0)
report(f'4 кольца: нейтральные есть ({found[4]})', found[4] > 0)
report('у всех нейтральных поперечный диполь = 0 (остаётся только осевой)', transverse_ok)
cfg = [(1, 0.), (-1, 0.)]
report('пара с полюсами в фазе нейтральна во всех A/B/C и плоскостях',
       all(maxsum(k, cfg, pl) < 1e-9 for k in 'ABC' for pl in PL))

## 6. Нелинейная теория: мёбиус-кольцо как голономия конечной дисклинации
В линейной теории скачок $b+\Omega\times x$ малый и линейный по $x$; поворот на $\pi$ невозможен.
В нелинейной калибровочной теории $SO(3)\ltimes T(3)$ (Kadić–Edelen, Katanaev–Volovich) локальный $b$, измеренный в параллельно переносимом репере, получает **голономию** $Q\in SO(3)$ при обходе петли, сцепленной с дисклинацией.
Закон $\psi=\chi/2$ даёт $b_{lab}(\theta)=R_z(\theta)\,R_n(\theta/2)\,b_{loc}$; при $\theta=2\pi$: $Q=R_{n}(\pi)$ — конечный поворот на $\pi$:
- $(r,\tau)$: $n=e_z$ — **клиновая** π-дисклинация вдоль оси спина (как $k=\tfrac12$ у Kleman–Friedel);
- $(r,z)$ и $(z,\tau)$: $n\perp e_z$ — **кручения** (twist) π-дисклинация.

Проверяем: $b(2\pi^-)=Q\,b(0^+)$ с $Q$ — ортогональной матрицей поворота на $\pi$, ось которой ожидаемая.

In [ ]:
def rot(n, a):
    n = np.asarray(n, float)/np.linalg.norm(n); K = np.array([[0,-n[2],n[1]],[n[2],0,-n[0]],[-n[1],n[0],0]])
    return np.eye(3) + np.sin(a)*K + (1-np.cos(a))*K@K
expected_axis = {'rt': [0,0,1], 'rz': [0,1,0], 'tz': [1,0,0]}   # оси при θ=0: e_z, e_τ(0)=y, e_r(0)=x
for pl in PL:
    n = expected_axis[pl]; Q = rot(n, np.pi)
    # базис плоскости (u,v) и ось n образуют тройку; проверяем голономию на векторах плоскости
    ok = True
    for psi0 in np.linspace(0, TW, 7):
        f0 = fr(np.array([1e-9])); f1 = fr(np.array([TW-1e-9])); u, v = PL[pl]
        b_start = np.cos(psi0)*f0[u][0] + np.sin(psi0)*f0[v][0]
        b_end = np.cos(psi0+np.pi)*f1[u][0] + np.sin(psi0+np.pi)*f1[v][0]   # ψ вырос на π за обход
        ok &= np.allclose(b_end, Q@b_start, atol=1e-6)
    report(f'{pl}: голономия = поворот на π вокруг {n} ({"клиновая" if pl=="rt" else "кручения"} π-дисклинация)', ok)
report('Q = R(π) ортогональна, det = 1, Q² = I (двузначный лифт в SU(2) → спин ½)',
       np.allclose(rot([0,0,1],np.pi)@rot([0,0,1],np.pi), np.eye(3)) and np.isclose(np.linalg.det(rot([0,0,1],np.pi)), 1))

## 7. Законы переноса (Fressengeas–Taupin 2020, ур. 88, 93–94)
$\dot\kappa^p=\theta\times V_\theta$, $\ \dot\alpha=-\mathrm{curl}\,\dot e^p+(\dot\kappa^p)^T-\mathrm{tr}(\dot\kappa^p)I$.
Для линейной дисклинации $\theta=t\otimes\Omega\,\delta$: $\dot\kappa^p\propto (t\times V)\otimes\Omega$.
- **B**: π-дисклинация голономии неподвижна (ось спина, $V_\theta=0$) — вращение $b$ есть поворот репера, $\dot\kappa^p=0$: кинематически свободно, без рождения дислокаций.
- **A, C**: шов бежит со скоростью $V=\omega R\,e_\tau$; дисклинация шва (перемычка N–S, вдоль $e_z$) движется поперёк своей линии: $t\times V\neq0$ ⇒ поток дислокаций $|\dot\alpha|\sim\pi\,\omega R$ на единицу длины, с сопротивлением $B_\theta$ — диссипация.
- Движение дефекта **вдоль** собственной линии ($V\parallel t$) потока не даёт.

In [ ]:
def kappa_dot(t, Omega, V):
    return np.outer(np.cross(t, V), Omega)      # ∝ θ×V для θ = t⊗Ω
phi = .7; om = 1.0
V_seam = om*R*fr(np.array([phi]))['t'][0]
kA = kappa_dot([0,0,1], np.pi*np.array([0,1,0]), V_seam)       # перемычка ∥ z, бежит по азимуту
kB = kappa_dot([0,0,1], np.pi*np.array([0,0,1]), np.zeros(3))  # клиновая π-дисклинация на оси, неподвижна
kL = kappa_dot(V_seam/np.linalg.norm(V_seam), np.pi*np.array([0,0,1]), V_seam)  # движение вдоль своей линии
report(f'A/C: бегущий шов рождает поток, |κ̇ᵖ| = {np.linalg.norm(kA):.2f} = π·ωR = {np.pi*om*R:.2f}', np.isclose(np.linalg.norm(kA), np.pi*om*R))
report('B: неподвижная дисклинация голономии, κ̇ᵖ = 0', np.allclose(kB, 0))
report('движение вдоль собственной линии: κ̇ᵖ = 0', np.allclose(kL, 0))

## 8. Сумма пары с параллельным переносом репера
При голономии $b_N$ и $b_S$ заданы в разных реперах. Ищем поворот $R(\beta)$ в плоскости $(u,v)$, который переводит $b_S$ в $-b_N$:
$\beta(\theta)=\psi_N-\psi_S$. Если бы $\beta$ был постоянным (или гладким), остаток можно было бы списать на связность.
Результат: $\beta=\pm90^\circ$ со скачком знака на полюсах (A) или на разрезе (B) — перенос гладкой связностью остаток **не убирает**;
нужна дисклинация с углом Франка $90^\circ$ между кольцами, меняющая знак — ещё один дефект.

In [ ]:
for k in 'AB':
    for phi in (0.3, 2.0):
        pN = {'A': m(TH-phi)/2+phi/2, 'B': -m(TH)/2+phi/2}[k]
        pS = {'A': m(TH-phi-np.pi)/2+phi/2, 'B': -m(TH-np.pi)/2+phi/2}[k]
        beta = np.degrees(np.angle(np.exp(1j*(pN-pS))))
        report(f'{k} φ={phi}: β(θ) = ±90° (не константа, скачет) → гладкий перенос не гасит остаток',
               np.allclose(np.abs(beta), 90, atol=1e-6) and beta.max() > 0 > beta.min())

## 9. Нейтральная частица из колец разного радиуса — поправка к теореме разд. 5
Мёбиус-закон $b(\theta)$ зависит только от азимута, не от радиуса. Поэтому вложенные кольца радиусов $R$ и $\rho R$ с одинаковым азимутом особых точек
и противоположными $s$ гасят друг друга по Бюргерсу, но поперечный диполь $\sum s_k r_k e_r(\varphi+\alpha_k)=2(1-\rho)R\,e_r(\varphi)\neq0$.
Пример: N$(R,0)$, S$(R,\pi)$, N$'(\rho R,\pi)$, S$'(\rho R,0)$ — **нейтральна и имеет вращающийся поперечный диполь**.

In [ ]:
def dipole(rings, phi):
    return sum(s*np.array([r*np.cos(phi+a), r*np.sin(phi+a), z]) for s, a, z, r in rings)
rho = 0.7
cfg4 = [(1, 0., H, R), (-1, np.pi, -H, R), (1, np.pi, .5*H, rho*R), (-1, 0., -.5*H, rho*R)]
neutral = all(np.allclose(sum(burgers(k, s, TH, p, pl, a) for s, a, z, r in cfg4), 0)
              for k in 'ABC' for pl in PL for p in np.linspace(0, 2*TW, 9))
pt = np.hypot(*dipole(cfg4, .7)[:2])
report('4 вложенных кольца: Σb = 0 для A/B/C во всех плоскостях', neutral)
report(f'и поперечный диполь |p⊥| = {pt:.2f} = 2(1−ρ)R = {2*(1-rho)*R:.2f} ≠ 0', np.isclose(pt, 2*(1-rho)*R))

## 10. Нелинейность: конечный поворот без деформации
Для поворота $F=R(\psi)$ линейная деформация $\varepsilon=\mathrm{sym}(F-I)=(1-\cos\psi)$ по модулю, при $\psi=\pi$ равна 2 — линейная теория «видит» огромную деформацию и энергию.
Тензор Альманси $e=\tfrac12(I-F^{-T}F^{-1})\equiv0$ для любого поворота. ⇒ поворот репера на $\pi$ вдоль мёбиус-кольца бесплатен по деформации **только** в нелинейной (эйлеровой) теории — как и утверждал автор.

In [ ]:
for d_ in (10, 45, 90, 180):
    a = np.radians(d_); F = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
    lin = .5*((F-np.eye(3)) + (F-np.eye(3)).T); Fi = np.linalg.inv(F); alm = .5*(np.eye(3) - Fi.T@Fi)
    report(f'ψ={d_}°: |ε_лин| = {np.abs(lin).max():.3f} (= 1−cos ψ), |e_Альманси| = 0',
           np.isclose(np.abs(lin).max(), 1-np.cos(a)) and np.abs(alm).max() < 1e-12)

## 11. Голономии в SU(2)
Подъём поворота на $\pi$ вокруг $e_z$: $U=-i\sigma_z$. Произведение по кольцам: одинаково закрученные пара — $U^2=-I$; мёбиус + анти-мёбиус — $+I$; четыре одинаковых — $U^4=+I$.
(Наблюдение о топологическом заряде системы; к статистике при повороте всей частицы на $2\pi$ напрямую не сводится.)

In [ ]:
U = np.array([[-1j, 0], [0, 1j]])
report('пара одинаково закрученных: U·U = −I', np.allclose(U@U, -np.eye(2)))
report('мёбиус + анти-мёбиус: U·U† = +I', np.allclose(U@U.conj().T, np.eye(2)))
report('4 одинаково закрученных кольца: U⁴ = +I', np.allclose(np.linalg.matrix_power(U, 4), np.eye(2)))

## 12. Четыре вложенных кольца: период и излучение
Конфигурация N$(R,0)$, S$(R,\pi)$, N$'(\rho R,\pi)$, S$'(\rho R,0)$.
- **Период.** Каждое мёбиус-кольцо при $\varphi\to\varphi+2\pi$ даёт $b\to-b$; значит, вся частица за $2\pi$ не возвращается, за $4\pi$ возвращается — критерий статьи даёт фермион, как и у пары.
- **Излучение.** Поперечный диполь особых точек $|p_\perp|=2(1-\rho)R$ против $2R$ у пары ⇒ мощность дипольного излучения $\propto(1-\rho)^2$, время высвечивания $\tau_4=\tau_2/(1-\rho)^2$ (при $\rho=0.7$ — в 11 раз дольше, но не бесконечно). Энергию и равновесный $\rho$ не считали.

In [ ]:
rho = 0.7
cfg4 = [(1, 0., H, R), (-1, np.pi, -H, R), (1, np.pi, .5*H, rho*R), (-1, 0., -.5*H, rho*R)]
for k in 'AB':
    ok = all(np.allclose(burgers(k, s, TH, .3+TW, 'rt', a), -burgers(k, s, TH, .3, 'rt', a)) and
             np.allclose(burgers(k, s, TH, .3+2*TW, 'rt', a), burgers(k, s, TH, .3, 'rt', a)) for s, a, z, r in cfg4)
    report(f'4 кольца {k}: φ+2π → −b у всех колец, φ+4π → b (период 4π)', ok)
p2 = np.hypot(*dipole([(1, 0., H, R), (-1, np.pi, -H, R)], .7)[:2]); p4 = np.hypot(*dipole(cfg4, .7)[:2])
report(f'|p⊥| пара = {p2:.2f} = 2R, 4 кольца = {p4:.2f}; τ₄/τ₂ = {(p2/p4)**2:.1f} = 1/(1−ρ)²', np.isclose((p2/p4)**2, 1/(1-rho)**2))

## 13. Статика в координатах Эйлера: прямая клиновая дисклинация (ось голономии) с тензором Альманси
Плоская деформация, полярная симметрия: точка $(r,\theta)$ текущей конфигурации пришла из $(R(r),\,\theta/\kappa)$, $\kappa=2\pi/(2\pi-\Omega)$.
$F^{-1}=\mathrm{diag}(R',\,R/(\kappa r))$, Альманси $e=\tfrac12(I-F^{-T}F^{-1})$, $\sigma=\lambda\,\mathrm{tr}e\,I+2\mu e$ (закон Гука на тензоре Альманси, как в Vyvod.ipynb), равновесие $\sigma_{rr}'+(\sigma_{rr}-\sigma_{\theta\theta})/r=0$, свободные границы $r_c$, $r_o$. Стрельба по $R(r_c)$.
**Результат:** при малых $\Omega$ совпадает с линейной теорией (проверка); с ростом $|\Omega|$ напряжения отходят от линейных на десятки процентов; при $|\Omega|\to\pi$ решение этим методом **не найдено** (стрельба нестабильна, траектории разрушаются). Модель «Гук на Альманси» при больших сжатиях, вероятно, теряет эллиптичность — нужен другой определяющий закон или метод (это вывод, не доказательство).

In [ ]:
mu_, lam_ = 1.0, 1.0
def _shoot(R0, kap, rc, ro, n=800):
    ett = .5*(1 - R0*R0/(kap*kap*rc*rc)); err = -lam_*ett/(lam_+2*mu_); q = 1 - 2*err
    if q <= 0: return None, None
    rs = np.linspace(rc, ro, n); h = rs[1]-rs[0]; Rv = R0; p = np.sqrt(q); out = np.empty((n, 2)); out[0] = Rv, p
    def f(r, Rv, p):
        e1 = .5*(1-p*p); e2 = .5*(1-Rv*Rv/(kap*kap*r*r)); s1 = lam_*(e1+e2)+2*mu_*e1; s2 = lam_*(e1+e2)+2*mu_*e2
        d2 = -(Rv/(kap*kap*r*r))*(p-Rv/r); return p, (lam_*d2+(s1-s2)/r)/((lam_+2*mu_)*p)
    for i in range(n-1):
        r = rs[i]; a1,b1 = f(r,Rv,p); a2,b2 = f(r+h/2,Rv+h/2*a1,p+h/2*b1); a3,b3 = f(r+h/2,Rv+h/2*a2,p+h/2*b2); a4,b4 = f(r+h,Rv+h*a3,p+h*b3)
        Rv += h/6*(a1+2*a2+2*a3+a4); p += h/6*(b1+2*b2+2*b3+b4)
        if not (np.isfinite(Rv) and np.isfinite(p)) or p <= 0: return rs, None
        out[i+1] = Rv, p
    return rs, out
def _st(rs, Y, kap):
    Rv, p = Y[:, 0], Y[:, 1]; e1 = .5*(1-p*p); e2 = .5*(1-Rv*Rv/(kap*kap*rs*rs))
    return lam_*(e1+e2)+2*mu_*e1, lam_*(e1+e2)+2*mu_*e2
def solve_wedge(Om, rc=.05, ro=1.):
    kap = TW/(TW-Om)
    def Fz(R0):
        rs, Y = _shoot(R0, kap, rc, ro); return None if Y is None else _st(rs, Y, kap)[0][-1]
    g = np.linspace(.05*rc*kap, 6*rc*kap, 120); v = [Fz(x) for x in g]
    br = [(g[i], g[i+1]) for i in range(119) if v[i] is not None and v[i+1] is not None and v[i]*v[i+1] < 0]
    if not br: return None
    a, b = br[0]; fa = Fz(a)
    for _ in range(40):
        c = (a+b)/2; fc = Fz(c)
        if fa*fc <= 0: b = c
        else: a, fa = c, fc
    rs, Y = _shoot((a+b)/2, kap, rc, ro); return kap, rs, Y
def lin_wedge(Om, rs, rc=.05, ro=1.):
    nu = lam_/(2*(lam_+mu_)); D = mu_*Om/(TW*(1-nu))
    C = np.linalg.solve(np.array([[1, 1/rc**2], [1, 1/ro**2]]), [-np.log(rc), -np.log(ro)])
    return D*(np.log(rs)+1+C[0]-C[1]/rs**2)
kap, rs, Y = solve_wedge(.02); r_ = np.abs(_st(rs, Y, kap)[1] - lin_wedge(.02, rs)).max()/np.abs(lin_wedge(.02, rs)).max()
report(f'Ω=0.02: нелинейное решение = линейному (отличие {r_:.3f})', r_ < .02)
for Om in (1.0, 2.0, -1.0, -2.0):
    s = solve_wedge(Om)
    if s: print(f'      Ω={Om:+.1f}: σθθ нелин/лин = {np.abs(_st(s[1], s[2], s[0])[1]).max()/np.abs(lin_wedge(Om, s[1])).max():.2f}')
report('Ω=±π: статическое решение методом стрельбы не найдено (открыто)', solve_wedge(np.pi) is None and solve_wedge(-np.pi) is None)

## 14. π-дисклинация с гиперупругим законом нео-Гука
Лагранжева форма (эквивалентна эйлеровой при правильной скорости, см. Vyvod.ipynb разд. 4). Отсчётная $(R,\Theta)$, $\Theta\in[0,2\pi-\Omega)$; текущая $r(R)$, $\theta=\kappa\Theta$, $\kappa=2\pi/(2\pi-\Omega)$.
$\lambda_r=r'$, $\lambda_\theta=\kappa r/R$, $J=\lambda_r\lambda_\theta$, плоская деформация:
$$W=\tfrac{\mu}{2}(\lambda_r^2+\lambda_\theta^2-2-2\ln J)+\tfrac{\lambda}{2}(\ln J)^2,$$
равновесие $\dfrac{d(R P_r)}{dR}=\kappa P_\theta$, $P_r=\mu(\lambda_r-\lambda_r^{-1})+\lambda\ln J/\lambda_r$, $P_\theta$ — аналогично; $P_r(R_c)=P_r(R_o)=0$.
В отличие от «Гука на Альманси» (разд. 13) энергия поликонвексна ($W\to\infty$ при $J\to0$), эллиптичность не теряется.
**Результат:** решение существует **во всём диапазоне до $\Omega=\pm\pi$**. Энергия $E(\pi)/E_{lin}=1.06$ (вырезанный клин), $E(-\pi)/E_{lin}=0.85$ (вставленный) — нелинейная поправка 6–15 %, знаковая асимметрия. Значит, неудача разд. 13 — свойство закона «Гук на Альманси», а не самой задачи.

In [ ]:
mu_h, lam_h = 1.0, 1.0
def _Pr(a, b): return mu_h*(a-1/a) + lam_h*np.log(a*b)/a
def _Pt(a, b): return mu_h*(b-1/b) + lam_h*np.log(a*b)/b
def _ar(P, b):
    lo, hi = 1e-6, 50.
    for _ in range(80):
        mm = .5*(lo+hi)
        if _Pr(mm, b) < P: lo = mm
        else: hi = mm
    return .5*(lo+hi)
def _sh(r0, kap, Rc, Ro, n):
    Rs = np.linspace(Rc, Ro, n); h = Rs[1]-Rs[0]; r = r0; S = 0.; out = [(r, S)]
    def f(Rv, r, S):
        b = kap*r/Rv; a = _ar(S/Rv, b); return a, kap*_Pt(a, b)
    for i in range(n-1):
        Rv = Rs[i]; k1 = f(Rv, r, S); k2 = f(Rv+h/2, r+h/2*k1[0], S+h/2*k1[1]); k3 = f(Rv+h/2, r+h/2*k2[0], S+h/2*k2[1]); k4 = f(Rv+h, r+h*k3[0], S+h*k3[1])
        r += h/6*(k1[0]+2*k2[0]+2*k3[0]+k4[0]); S += h/6*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        if not (np.isfinite(r) and r > 0): return Rs, None
        out.append((r, S))
    return Rs, np.array(out)
def neo_wedge(Om, Rc=.05, Ro=1., n_fin=1200):
    kap = TW/(TW-Om)
    F = lambda r0: (lambda Rs, Y: None if Y is None else Y[-1, 1])(*_sh(r0, kap, Rc, Ro, 300))
    g = np.geomspace(.002*Rc, 6*Rc, 70); v = [F(x) for x in g]
    br = [(g[i], g[i+1]) for i in range(69) if v[i] is not None and v[i+1] is not None and v[i]*v[i+1] < 0]
    if not br: return None
    a, b = br[0]; fa = F(a)
    for _ in range(40):
        c = np.sqrt(a*b); fc = F(c)
        if fa*fc <= 0: b = c
        else: a, fa = c, fc
    Rs, Y = _sh(np.sqrt(a*b), kap, Rc, Ro, n_fin)
    r = Y[:, 0]; al = np.gradient(r, Rs); bt = kap*r/Rs; J = al*bt
    W = mu_h/2*(al*al+bt*bt-2-2*np.log(J)) + lam_h/2*np.log(J)**2
    return (TW-Om)*np.trapezoid(W*Rs, Rs), J
def E_lin_w(Om, Rc=.05, Ro=1.):
    nu_ = lam_h/(2*(lam_h+mu_h)); x = Rc/Ro
    return mu_h*Om**2*Ro**2/(16*np.pi*(1-nu_))*(1 - x**2 - 4*x**2*np.log(x)**2/(1-x**2))
E, J = neo_wedge(.05); report(f'Ω=0.05: E_нео/E_лин = {E/E_lin_w(.05):.3f} (≈1, проверка метода)', abs(E/E_lin_w(.05)-1) < .01)
for Om in (np.pi, -np.pi):
    s = neo_wedge(Om)
    report(f'Ω={Om:+.2f}: решение есть, E/E_лин = {s[0]/E_lin_w(Om):.3f}, J∈[{s[1].min():.2f},{s[1].max():.2f}] (J>0 всюду)',
           s is not None and s[1].min() > 0)

## 15. Энергия 4-кольцевой частицы и равновесный $\rho$ (оценка, линейная теория)
Энергия набора петель — двойной контурный интеграл (формула Блина, линейная изотропная упругость; **записана по памяти, сверить с Hirth–Lothe**), ядро регуляризовано $\sqrt{R^2+a^2}$:
$$W_{12}=-\frac{\mu}{2\pi}\oint\!\!\oint\frac{(b_1\times b_2)\cdot(dl_1\times dl_2)}{R}+\frac{\mu}{4\pi}\oint\!\!\oint\frac{(b_1\cdot dl_1)(b_2\cdot dl_2)}{R}+\frac{\mu}{4\pi(1-\nu)}\oint\!\!\oint(b_1\times dl_1)\cdot\mathbf T\cdot(b_2\times dl_2),\ \ T_{ij}=\partial_i\partial_jR .$$
Для мёбиус-колец $b$ меняется вдоль линии — формула применяется как оценка (строго она для дефектов Вольтерры).
**Результат:** $E_4(\rho)$ монотонно растёт с $\rho$ (собственная энергия внутренних колец $\propto\rho R$), взаимодействие пар мало и положительно. **Статического равновесия по $\rho$ нет — внутренние кольца стягиваются ($\rho\to0$)**, после чего остаётся ненейтральная пара. Устойчивость возможна только динамически (центробежный барьер от прецессии, как $d_{eq}=J^2/mA$ у пары) — не посчитано.

In [ ]:
mu_b, nu_b, a_c = 1.0, .25, .15
def ring_b(kind, pl, s, alpha, z, rad, phi, n=120):
    th = (np.arange(n)+.5)*TW/n; f = fr(th); u, v = PL[pl]
    p = m(th-phi-alpha)/2 + phi/2 if kind == 'A' else -m(th-alpha)/2 + phi/2
    bv = s*(np.cos(p)[:, None]*f[u] + np.sin(p)[:, None]*f[v])
    x = np.stack([rad*np.cos(th), rad*np.sin(th), np.full(n, z)], 1); dl = rad*TW/n*f['t']
    return x, dl, bv
def W12(A, B, same=False):
    x1, d1, b1 = A; x2, d2, b2 = B
    D = x1[:, None, :] - x2[None, :, :]; Rr = np.sqrt((D**2).sum(-1) + a_c**2)
    t1 = -(mu_b/TW)*np.einsum('ijk,ijk->ij', np.cross(b1[:, None, :], b2[None, :, :]), np.cross(d1[:, None, :], d2[None, :, :]))/Rr
    t2 = (mu_b/(2*TW))*np.outer((b1*d1).sum(1), (b2*d2).sum(1))/Rr
    c1 = np.cross(b1, d1); c2 = np.cross(b2, d2)
    t3 = (mu_b/(2*TW*(1-nu_b)))*((c1@c2.T)/Rr - np.einsum('ijk,ik->ij', D, c1)*np.einsum('ijk,jk->ij', D, c2)/Rr**3)
    w = (t1+t2+t3).sum(); return .5*w if same else w
def E_rings(rr): return sum(W12(rr[i], rr[j], i == j) for i in range(len(rr)) for j in range(i, len(rr)))
def cfg4b(kind, pl, rho, phi):
    return [ring_b(kind, pl, 1, 0., H, R, phi), ring_b(kind, pl, -1, np.pi, -H, R, phi),
            ring_b(kind, pl, 1, np.pi, .5*H, rho*R, phi), ring_b(kind, pl, -1, 0., -.5*H, rho*R, phi)]
ph6 = np.linspace(0, 2*TW, 6, endpoint=False); rhos = (.1, .3, .5, .7, .9)
for kind, pl in (('B', 'rt'), ('A', 'rz')):
    E = [np.mean([E_rings(cfg4b(kind, pl, r, p)) for p in ph6]) for r in rhos]
    print('      ', kind, pl, ' E4(ρ):', ', '.join(f'{r}:{e:.2f}' for r, e in zip(rhos, E)))
    report(f'{kind} {pl}: E4 монотонно растёт с ρ → статического равновесия нет (стягивание ρ→0)', all(np.diff(E) > 0))

## 16. Сверка формулы энергии петель (Блин 1955) с Hirth–Lothe
Формула разд. 15 названа у Lazar–Kirchner 2017 (arXiv:1706.00828) формулой Блина для энергии взаимодействия двух петель.
Проверка на собственной энергии круговой петли: в Hirth–Lothe $E\approx k\,\mu b^2R\,[\ln(8R/\rho)-c]$, $k_{\rm prism}=\frac{1}{2(1-\nu)}$, $k_{\rm glide}=\frac{2-\nu}{4(1-\nu)}$.
Сравниваем наклон $d(E/R)/d\ln R$. **Результат:** оба наклона завышены на одинаковые ~9.5 % — относительные веса членов верны, разница — дискретизация и регуляризация ядра на ограниченном диапазоне $R$. Выводы разд. 15 (монотонность по $\rho$) от общего множителя не зависят.

In [ ]:
def loop_c(Rv, bvec, n=400):
    th = (np.arange(n)+.5)*TW/n
    x = np.stack([Rv*np.cos(th), Rv*np.sin(th), 0*th], 1); t = np.stack([-np.sin(th), np.cos(th), 0*th], 1)
    return x, Rv*TW/n*t, np.broadcast_to(np.array(bvec, float), (n, 3)).copy()
a_save = a_c; a_c = .05; ratios = []
for nm, bv, kHL in (('призм.', [0, 0, 1], 1/(2*(1-nu_b))), ('glide', [1, 0, 0], (2-nu_b)/(4*(1-nu_b)))):
    Rs_ = np.array([2., 4., 8.]); E_ = np.array([W12(loop_c(r, bv), loop_c(r, bv), True) for r in Rs_])
    sl = np.polyfit(np.log(Rs_), E_/Rs_, 1)[0]; ratios.append(sl/kHL)
    print(f'      {nm}: наклон {sl:.4f}, Hirth–Lothe {kHL:.4f}, отношение {sl/kHL:.3f}')
a_c = a_save
report('отношения для призм. и glide петель совпадают (структура формулы верна)', abs(ratios[0]-ratios[1]) < .01)
report('абсолютный множитель в пределах 10 %', all(abs(r-1) < .1 for r in ratios))

## 17. Динамический барьер для 4-кольцевой частицы
Статическая энергия (разд. 15) почти линейна: $E(\rho)\approx E_0+A\rho$. Если внутренняя пара сохраняет момент импульса прецессии $L$, а её масса $\propto$ длине ($M=k\rho$), то момент инерции $I=k R^2\rho^3$ и
$$E_{\rm eff}(\rho)=E_0+A\rho+\frac{B}{\rho^3},\quad B=\frac{L^2}{2kR^2},\qquad \rho^*=\Big(\frac{3B}{A}\Big)^{1/4},\quad E''(\rho^*)=\frac{12B}{\rho^{*5}}>0 .$$
Равновесие **существует и устойчиво**, если $B<A/3$ (иначе $\rho^*>1$ — внутренние кольца выходят наружу). Допущения: сохранение $L$ внутренней пары, масса $\propto$ длине, $\omega$ не зависит от $\rho$ — не выведены.

In [ ]:
E_st = [np.mean([E_rings(cfg4b('B', 'rt', r, p)) for p in ph6]) for r in rhos]
Acoef, E0 = np.polyfit(rhos, E_st, 1)
report(f'E(ρ) ≈ {E0:.2f} + {Acoef:.2f}ρ (откл. {np.abs(np.polyval([Acoef, E0], rhos)-E_st).max():.2f})', np.abs(np.polyval([Acoef, E0], rhos)-E_st).max() < .5)
rr_ = np.linspace(.03, 1.5, 40000)
for Bc in (.01, .1, 1.):
    Ee = E0 + Acoef*rr_ + Bc/rr_**3; num = rr_[np.argmin(Ee)]; an = (3*Bc/Acoef)**.25
    report(f'B={Bc}: ρ* = {num:.4f} = (3B/A)^¼ = {an:.4f}, устойчиво', abs(num-an) < 1e-3 and 12*Bc/an**5 > 0)

## Итог
| результат | статус |
|---|---|
| A, B — период 4π; C — 2π | OK |
| противофаза ⇒ $|b_N+b_S|=\sqrt2 b_0$; формула $2b_0\max(|\cos\Delta/4|,|\sin\Delta/4|)$ | OK |
| одной линией Вольтерры не гасится | OK |
| нейтральность ⇔ попарное гашение полюсов ⇒ нет поперечного диполя; 3 кольца — никогда | OK |
| в нелинейной теории мёбиус-кольцо = петля, сцепленная с конечной π-дисклинацией (голономия) | OK (кинематика) |
| по законам переноса B свободна, A/C порождают поток дислокаций | OK (оценка) |
| перенос репера не гасит остаток пары (β = ±90° со скачком) | OK |
| нейтральность + поперечный диполь возможны для вложенных колец разного радиуса | OK (исправляет разд. 5) |
| поворот на π: ε_лин = 2, Альманси = 0 | OK |
| SU(2): пара −I, 4 кольца +I | OK (наблюдение) |
| 4 вложенных кольца: период 4π; τ₄/τ₂ = 1/(1−ρ)² | OK |
| эйлерова статика с Альманси: малые Ω = линейная теория; Ω=±π — решение не найдено | OK / открыто |
| π-дисклинация, нео-Гук: решение до Ω=±π; E/E_лин = 1.06 / 0.85 | OK |
| 4 кольца: E(ρ) растёт, статического равновесия нет (нужен динамический барьер) | OK (оценка; формула Блина по памяти) |
| формула Блина: структура совпадает с Hirth–Lothe, множитель ±10 % | OK |
| динамический барьер: ρ* = (3B/A)^¼ устойчиво при B < A/3 | OK (при допущениях) |